# Data collection (raw)

Sources: **yfinance**, **FRED** (no API key required) and **Cboe** (VIX).
This notebook only downloads the data and saves it to the `data/raw/` folder.
Missing values (NaN) are **not filled** here; cleaning is done in a separate step (`data/processed/`).


In [9]:
# !pip install yfinance openpyxl
from datetime import timedelta
from pathlib import Path
from io import StringIO
import requests
import time

import pandas as pd
import yfinance as yf


# Settings 
START = "2000-01-01"
END = "2026-11-01"     # inclusive

# Tickers for yfinance
TICKERS = ["^GSPC", "^IXIC", "^DJI", "^VIX"]

# FRED series (fredgraph.csv endpoint, no API key needed)
FRED_SERIES = {
    "VIXCLS":   "VIX closing value",
    "DGS2":     "2-year Treasury yield",
    "DGS10":    "10-year Treasury yield",
    "DFF":      "Federal Funds Effective Rate",
    "CPIAUCSL": "CPI (monthly)",
}

CBOE_VIX_URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"

SAVE_CSV = True
SAVE_EXCEL = False          # set to True to also save .xlsx (requires openpyxl)

# Paths 
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", RAW_DIR.resolve())


def save_raw(df: pd.DataFrame, name: str) -> None:
    """Save a DataFrame to data/raw/. The Date index is kept as a column."""
    if SAVE_CSV:
        df.to_csv(RAW_DIR / f"{name}.csv", index=True)
    if SAVE_EXCEL:
        df.to_excel(RAW_DIR / f"{name}.xlsx", index=True)
    print(f"{name}: {df.shape[0]} rows, {df.index.min().date()} -> {df.index.max().date()}, NaN: {int(df.isna().sum().sum())}")


Output folder: C:\Users\User\Documents\Diploma\data\raw


## 1. yfinance

In [10]:
# yfinance treats `end` as exclusive, so add one day
YF_END = (pd.Timestamp(END) + timedelta(days=1)).strftime("%Y-%m-%d")

for tk in TICKERS:
    df = yf.download(tk, start=START, end=YF_END, auto_adjust=False, progress=False)

    # Recent yfinance versions return MultiIndex columns, so flatten them
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    if df.empty:
        print(f"{tk}: no data returned")
        continue

    df.index = pd.to_datetime(df.index)
    df.index.name = "Date"
    df = df.loc[START:END]

    name = tk.replace("^", "").replace("=", "_") + "_yf_raw"
    save_raw(df, name)
    

GSPC_yf_raw: 6748 rows, 2000-01-03 -> 2026-10-30, NaN: 0
IXIC_yf_raw: 6748 rows, 2000-01-03 -> 2026-10-30, NaN: 0
DJI_yf_raw: 6748 rows, 2000-01-03 -> 2026-10-30, NaN: 0
VIX_yf_raw: 6750 rows, 2000-01-03 -> 2026-10-30, NaN: 0


## 2. FRED
In some FRED CSV files missing values are written as `.`, which is why `na_values="."` is used when reading them.

In [11]:
# FRED marks missing values with "." in some CSVs, hence na_values=["."]
# Set a clear client identifier instead of a generic browser string
HEADERS = {"User-Agent": "SP500-Forecasting-Project/1.0"}

for sid, desc in FRED_SERIES.items():
  start_time = time.time()
  url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}"
  print(f"Downloading {sid} ({desc})...")

  try:
    # Use a timeout so a stalled connection cannot hang the kernel indefinitely
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text), na_values=["."])

    # The first column is the date (named DATE or observation_date)
    df = df.rename(columns={df.columns[0]: "Date", df.columns[1]: sid})
    df["Date"] = pd.to_datetime(df["Date"])
    df[sid] = pd.to_numeric(df[sid], errors="coerce")

    df = df.set_index("Date").loc[START:END]
    save_raw(df, f"{sid}_fred_raw")

    elapsed = time.time() - start_time
    print(f"  Saved {sid}_fred_raw ({len(df)} rows, {elapsed:.2f}s)")
  except Exception as e:
    elapsed = time.time() - start_time
    print(f"  Failed for {sid} ({elapsed:.2f}s): {e}")

  # Brief pause between requests to avoid hammering the server
  time.sleep(0.3)

VIXCLS_fred_raw: 7000 rows, 2000-01-03 -> 2026-10-30, NaN: 218
  Saved VIXCLS_fred_raw (7000 rows, 0.41s)
DGS2_fred_raw: 7000 rows, 2000-01-03 -> 2026-10-30, NaN: 289
  Saved DGS2_fred_raw (7000 rows, 0.58s)
DGS10_fred_raw: 7000 rows, 2000-01-03 -> 2026-10-30, NaN: 289
  Saved DGS10_fred_raw (7000 rows, 0.44s)
DFF_fred_raw: 9802 rows, 2000-01-01 -> 2026-11-01, NaN: 0
  Saved DFF_fred_raw (9802 rows, 0.54s)
CPIAUCSL_fred_raw: 322 rows, 2000-01-01 -> 2026-10-01, NaN: 1
  Saved CPIAUCSL_fred_raw (322 rows, 0.22s)


## 3. Cboe (VIX OHLC)

In [12]:
try:
    vix = pd.read_csv(CBOE_VIX_URL, storage_options=HEADERS)
    vix.columns = [c.strip().title() for c in vix.columns]     # DATE -> Date, CLOSE -> Close, ...
    vix["Date"] = pd.to_datetime(vix["Date"])
    vix = vix.set_index("Date").sort_index().loc[START:END]
    save_raw(vix, "VIX_cboe_raw")
except Exception as e:
    print("Cboe download failed; use FRED VIXCLS or yfinance ^VIX instead:", e)
    

VIX_cboe_raw: 6782 rows, 2000-01-03 -> 2026-10-30, NaN: 0


## 4. Quick check

In [13]:
# Summary of all saved raw files
rows = []
for f in sorted(RAW_DIR.glob("*.csv")):
    d = pd.read_csv(f, index_col="Date", parse_dates=True)
    rows.append({
        "file": f.name,
        "rows": len(d),
        "start": d.index.min().date(),
        "end": d.index.max().date(),
        "NaN": int(d.isna().sum().sum()),
    })
pd.DataFrame(rows)


,file,rows,start,end,NaN
0,CPIAUCSL_fred_raw.csv,322,2000-01-01,2026-10-01,1
1,DFF_fred_raw.csv,9802,2000-01-01,2026-11-01,0
2,DGS10_fred_raw.csv,7000,2000-01-03,2026-10-30,289
3,DGS2_fred_raw.csv,7000,2000-01-03,2026-10-30,289
4,DJI_yf_raw.csv,6748,2000-01-03,2026-10-30,0
5,GSPC_yf_raw.csv,6748,2000-01-03,2026-10-30,0
6,IXIC_yf_raw.csv,6748,2000-01-03,2026-10-30,0
7,VIX_cboe_raw.csv,6782,2000-01-03,2026-10-30,0
8,VIX_yf_raw.csv,6750,2000-01-03,2026-10-30,0
9,VIXCLS_fred_raw.csv,7000,2000-01-03,2026-10-30,218
